# micrograd 4: Neuron → Layer → MLP, and training

Karpathy, *Zero to Hero* #1, from ~1:43 to the end. **Skip 1:39–1:43 (PyTorch): that's Sunday.**

Goal: stop building neurons by hand. Write three small classes, give them a dataset, and let `backward()` plus gradient descent do the fitting.

**Rules:** pause before each step he does and do it first. Type everything. Predict before you run.

**Before anything:** the Friday opener and the micrograd 3 reflection, both at the bottom of `03_backward.ipynb`.

## Block 0: carry-over (2 min)
`Value` as it stood at the end of Wednesday, plus `__rsub__`. Compare with your codespace version: every `_backward` must use `+=`. Run it.

Quick check before running: why does `__neg__` need no `_backward`?

In [ ]:
import math, random
from viz import draw_dot

class Value:
    def __init__(self, data, _children=(), _op='', label=''):
        self.data = data
        self.grad = 0.0
        self._backward = lambda: None
        self._prev = set(_children)
        self._op = _op
        self.label = label

    def __repr__(self):
        return f"Value(data={self.data})"

    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data + other.data, (self, other), '+')
        def _backward():
            self.grad += 1.0 * out.grad
            other.grad += 1.0 * out.grad
        out._backward = _backward
        return out

    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data * other.data, (self, other), '*')
        def _backward():
            self.grad += other.data * out.grad
            other.grad += self.data * out.grad
        out._backward = _backward
        return out

    def __pow__(self, other):
        assert isinstance(other, (int, float)), "only int/float powers"
        out = Value(self.data ** other, (self,), f'**{other}')
        def _backward():
            self.grad += other * self.data ** (other - 1) * out.grad
        out._backward = _backward
        return out

    def exp(self):
        out = Value(math.exp(self.data), (self,), 'exp')
        def _backward():
            self.grad += out.data * out.grad
        out._backward = _backward
        return out

    def tanh(self):
        t = math.tanh(self.data)
        out = Value(t, (self,), 'tanh')
        def _backward():
            self.grad += (1 - t**2) * out.grad
        out._backward = _backward
        return out

    def __neg__(self): return self * -1
    def __sub__(self, other): return self + (-other)
    def __truediv__(self, other): return self * other**-1
    def __radd__(self, other): return self + other
    def __rmul__(self, other): return self * other
    def __rsub__(self, other): return other + (-self)

    def backward(self):
        order, visited = [], set()
        def build(v):
            if v not in visited:
                visited.add(v)
                for child in v._prev:
                    build(child)
                order.append(v)
        build(self)
        self.grad = 1.0
        for node in reversed(order):
            node._backward()

## Block 2: `Neuron` (~1:43 to ~1:49, 25 min)
A neuron with `nin` inputs holds `nin` weights and one bias. Called on an input list `x`, it returns tanh(w · x + b).

1. **Before he does:** finish `__call__` below. One line for the weighted sum, one for the tanh.
2. If you use `sum(...)`: Python's `sum` starts at the plain number `0`. What does `0 + Value(...)` need from the class? Where could you start the sum instead, to save a node?
3. Predict: for `Neuron(2)` called on `[2.0, 3.0]`, how many nodes in the graph (leaves included)? Then `draw_dot` and count.
4. Run the cell twice. Why is the output different each time? Is that a problem?

In [ ]:
class Neuron:
    def __init__(self, nin):
        self.w = [Value(random.uniform(-1, 1)) for _ in range(nin)]
        self.b = Value(random.uniform(-1, 1))

    def __call__(self, x):
        # w · x + b, then tanh
        ___

# predict: node count = ___
x = [2.0, 3.0]
n = Neuron(2)
out = n(x)
print(out)
draw_dot(out)

## Block 3: `Layer` and `MLP` (~1:49 to ~1:56, 20 min)
- A **Layer** is a list of `nout` neurons, each seeing the same `nin` inputs. Called on `x`, it returns the list of their outputs (or the single output if there's only one: why is that convenient?).
- An **MLP** chains layers: `MLP(3, [4, 4, 1])` means 3 inputs → 4 neurons → 4 neurons → 1 output. Each layer's output list is the next layer's input.

1. **Before he does:** write both classes.
2. **On paper first:** how many parameters (weights + biases) does `MLP(3, [4, 4, 1])` have? Show the working layer by layer.
3. Add `parameters()` to all three classes (a flat list of every `Value` that training may change). Check `len(...)` against your paper count.
4. Interview-style: suppose every weight in a layer started at the **same** value (say 0.5) instead of random. What would be true of those neurons' outputs and gradients, and would training ever make them different?

In [ ]:
class Layer:
    ___

class MLP:
    ___

# on paper: parameters = ___ + ___ + ___ = ___

In [ ]:
mlp = MLP(3, [4, 4, 1])
print(mlp([2.0, 3.0, -1.0]))
print(len(mlp.parameters()))

## Break (5 min)

## Block 4: loss and one step by hand (~1:56 to ~2:03, 25 min)
Four inputs, four targets. We want `mlp(xs[i])` close to `ys[i]` for each i.

1. Loss = sum over the four examples of (prediction − target)². **Before running:** what's the smallest possible loss, and the largest? (What range can tanh output, and what are the targets?)
2. Why square the difference rather than just add the differences up?
3. `loss.backward()`. Pick `mlp.layers[0].neurons[0].w[0]`. Read its `grad`. If you **increase** that weight slightly, does the loss go up or down?
4. One step by hand for *every* parameter: `p.data += -0.05 * p.grad`. Why the minus sign? Predict: will the loss go down? By roughly how much (use the linear approximation idea from session 2)?
5. Recompute the loss (fresh forward pass) and check. Then: what happens if you call `loss.backward()` again **without** recomputing? (Wednesday's 3× result.)

In [ ]:
xs = [
    [2.0, 3.0, -1.0],
    [3.0, -1.0, 0.5],
    [0.5, 1.0, 1.0],
    [1.0, 1.0, -1.0],
]
ys = [1.0, -1.0, -1.0, 1.0]  # targets

mlp = MLP(3, [4, 4, 1])
ypred = [mlp(x) for x in xs]
loss = ___
print(loss)

In [ ]:
# backward, read one gradient. predict: increasing w[0] makes the loss go ___

In [ ]:
# one step by hand. predict: new loss ≈ ___

## Block 5: the training loop (~2:03 to the end, 25 min)
1. **Before he does:** write the loop: forward pass → loss → backward → update, repeated 20 times, printing the step and loss.
2. **Before running it**, read your loop with Wednesday's `backward()`-twice result in mind. Is anything missing? Write your answer in a comment, then run.
3. When the loss is small, print `ypred`. Do they match `ys`?
4. Learning rate: try 0.01, 0.1 and 0.5 on a **fresh** `MLP` each time. Predict which works best first. What does too big look like? Too small?
5. Karpathy admits a bug in his own loop on camera. If you've already avoided it, say which line saved you. Then put the bug *in* deliberately and rerun at 0.1 and at 0.5. At one of them the buggy loop may look *better*. Why is it still wrong, and why is "the loss went down" not enough to show your code is right?

In [ ]:
# Is anything missing? ___

mlp = MLP(3, [4, 4, 1])
for k in range(20):
    ___

In [ ]:
# learning rate. predict best of 0.01 / 0.1 / 0.5: ___

## Wrap (10 min)
**CS × Phil:** your network has 41 parameters and you trained it on 4 examples. The loss is near 0.

- Has it *learnt* anything, or just stored four answers? What test would tell you? (Try it: invent a fifth input and ask the network. Does the answer mean anything?)
- Link to Aaronson §4 and your zip test: a lookup table "passes" on stored inputs. You said compression only counts when it *generalises*. Is 41 parameters for 4 data points compression at all?

**Reflection:** learned / surprised / question it opened, for the activity log.

Then tick the README box and:
`git add -A && git commit -m "micrograd 4: MLP + training" && git push`